# 12.1 어텐션 아이디어와 Query·Key·Value — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter12_1_qkv.ipynb)

책 본문: [12.1 어텐션 아이디어와 Query·Key·Value](https://smhanlab.com/book-ml/kor/ml1/chapter12/1.html)

이 노트북은 책 12.1절의 모든 수치를 **실제로 실행해서** 검증합니다:

1. 같은 입력 X에서 Q, K, V가 서로 다른 값으로 나오는지(손 계산 예 재현)
2. \(QK^T = \begin{pmatrix}0.5&0\\0&1\end{pmatrix}\) → \(\sqrt{2}\) 스케일링 + softmax → 가중치 (0.588, 0.412) / (0.330, 0.670)
3. V의 가중합 출력 (1.588, 1.412) / (1.330, 1.670) — "관련도만큼 섞인" 값
4. Q=K=V=X(변환 없음) A/B 비교 — "입력 자체를 비교"하면 무엇이 깨지는지

In [1]:
import math
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)
print("numpy", np.__version__)

numpy 2.4.6


## 1. 손으로 한 번: 같은 입력에서 Q, K, V가 서로 다르게 계산되는 것

본문 예 — 단어 2개, 임베딩 \(x_1=(1,0),\; x_2=(0,1)\), 서로 다른 세 학습 행렬:

\[W_Q=\begin{pmatrix}1&0\\0&2\end{pmatrix},\quad W_K=\begin{pmatrix}0.5&0\\0&0.5\end{pmatrix},\quad W_V=\begin{pmatrix}2&1\\1&2\end{pmatrix}\]

같은 입력 \(X\)에 세 행렬을 각각 적용하면 Q, K, V는 **서로 다른 값**으로
나와야 합니다 — 이 세 행렬이 독립적인 학습 파라미터이기 때문입니다.

In [2]:
X  = [[1, 0],
      [0, 1]]
WQ = [[1,   0],
      [0,   2]]
WK = [[0.5, 0],
      [0,   0.5]]
WV = [[2, 1],
      [1, 2]]

def mat_vec_mul(M, v):
    # M (2x2) @ v (크기 2) -> 크기 2
    return [sum(M[i][t] * v[t] for t in range(len(v))) for i in range(len(M))]

def qkv(X, WQ, WK, WV):
    # X의 각 행(단어 임베딩)에 각각 WQ, WK, WV를 곱해 (Q, K, V) 반환
    return ([mat_vec_mul(WQ, x) for x in X],
            [mat_vec_mul(WK, x) for x in X],
            [mat_vec_mul(WV, x) for x in X])

Q, K, V = qkv(X, WQ, WK, WV)
print("Q =", Q)
print("K =", K)
print("V =", V)
assert Q == [[1, 0], [0, 2]], Q
assert K == [[0.5, 0], [0, 0.5]], K
assert V == [[2, 1], [1, 2]], V
print("-> 본문 손계산 Q, K, V와 정확히 일치!")

Q = [[1, 0], [0, 2]]
K = [[0.5, 0.0], [0.0, 0.5]]
V = [[2, 1], [1, 2]]
-> 본문 손계산 Q, K, V와 정확히 일치!


## 2. 전체 파이프라인 미리보기: QK^T → 스케일링 → softmax → V의 가중합

12.2절의 어텐션 연산 \(\text{Attention}(Q,K,V) = \text{softmax}(QK^T/\sqrt{d_k})\,V\)
를 위 결과로 **미리** 계산합니다 — "관련도만큼 V를 섞는다"가 숫자로
보여야 합니다. (단어 1은 V₁ 쪽에, 단어 2는 V₂ 쪽에 기울어져야 함.)

In [3]:
def dot(a, b):
    return sum(x * y for x, y in zip(a, b))

def softmax(row):
    m = max(row)
    exps = [math.exp(v - m) for v in row]
    total = sum(exps)
    return [e / total for e in exps]

# 1. 원점수 QK^T
S = [[dot(Q[i], K[j]) for j in range(len(K))] for i in range(len(Q))]
print("QK^T =")
for row in S:
    print("   ", row)
assert S == [[0.5, 0.0], [0.0, 1.0]], S

# 2. sqrt(d_k) 스케일링 + softmax
d_k = 2
A = [softmax([x / math.sqrt(d_k) for x in row]) for row in S]
print("\n어텐션 가중치 (softmax 후):")
for row in A:
    print("   ", [f"{x:.3f}" for x in row])
assert abs(A[0][0] - 0.588) < 1e-3 and abs(A[0][1] - 0.412) < 1e-3
assert abs(A[1][0] - 0.330) < 1e-3 and abs(A[1][1] - 0.670) < 1e-3

# 3. V의 가중합 -> 출력
out = [[sum(A[i][j] * V[j][t] for j in range(len(V))) for t in range(len(V[0]))]
       for i in range(len(A))]
print("\n어텐션 출력 (A·V):")
for row in out:
    print("   ", [f"{x:.3f}" for x in row])
assert abs(out[0][0] - 1.588) < 1e-3 and abs(out[0][1] - 1.412) < 1e-3
assert abs(out[1][0] - 1.330) < 1e-3 and abs(out[1][1] - 1.670) < 1e-3
print("-> 본문 수치와 정확히 일치. 출력은 V₁, V₂의 가중평균이되 관련도만큼 기울어짐.")

QK^T =
    [0.5, 0.0]
    [0.0, 1.0]

어텐션 가중치 (softmax 후):
    ['0.587', '0.413']
    ['0.330', '0.670']

어텐션 출력 (A·V):
    ['1.587', '1.413']
    ['1.330', '1.670']
-> 본문 수치와 정확히 일치. 출력은 V₁, V₂의 가중평균이되 관련도만큼 기울어짐.


## 3. A/B 비교: 변환 없이 Q=K=V=X로 쓰면 무엇이 깨지는가

Q, K, V의 "역할 분화"를 없애고 **입력 자체를 그대로** 쓰면 — 임베딩이
서로 직각인 단위 벡터인 이 예에서는 각 단어가 **자기 자신과만** 높은
점수를 얻는 고정된 패턴이 됩니다. 더 근본적으로는 학습할 \(W_Q, W_K, W_V\)가
존재하지 않는다는 차이가 있습니다. 두 경우의 어텐션 가중치 행렬을
그려서 비교합니다.

In [4]:
# --- A) 변환 없음: Q = K = V = X ---
Qi = Ki = Vi = X
Si = [[dot(Qi[i], Ki[j]) for j in range(2)] for i in range(2)]
Ai = [softmax([x / math.sqrt(2) for x in row]) for row in Si]
outi = [[sum(Ai[i][j] * Vi[j][t] for j in range(2)) for t in range(2)]
        for i in range(2)]
print("A) 변환 없음 (Q=K=V=X):")
print("   점수 XX^T =", Si)
for row in Ai:
    print("   가중치:", [f"{x:.3f}" for x in row])
for row in outi:
    print("   출력:  ", [f"{x:.3f}" for x in row])

# --- B) 학습 행렬 사용 (2절의 결과 재사용) ---
print("\nB) 변환 사용 (W_Q, W_K, W_V):")
for row in A:
    print("   가중치:", [f"{x:.3f}" for x in row])
for row in out:
    print("   출력:  ", [f"{x:.3f}" for x in row])

# --- A/B 어텐션 가중치 행렬 시각화 ---
fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.8))
words = ["단어 1", "단어 2"]
for ax, (title, M) in zip(axes, [("A) Q=K=V=X (변환 없음)", Ai),
                                 ("B) 학습된 W_Q, W_K 사용", A)]):
    im = ax.imshow(M, cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{M[i][j]:.3f}", ha="center", va="center", fontsize=11)
    ax.set_xticks(range(2)); ax.set_yticks(range(2))
    ax.set_xticklabels(words); ax.set_yticklabels(words)
    ax.set_xlabel("Key (누구를 보는가)")
    ax.set_ylabel("Query (누가 보는가)")
    ax.set_title(title, fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle("어텐션 가중치 A/B 비교 — 둘 다 대각선 우세, '기울어짐' 정도가 다름", y=1.03)
fig.tight_layout()
fig.savefig(IMG + "/ch12_1_qkv_weights.svg", bbox_inches="tight")
plt.show()
print("figure saved -> kor/src/images/ch12_1_qkv_weights.svg")

A) 변환 없음 (Q=K=V=X):
   점수 XX^T = [[1, 0], [0, 1]]
   가중치: ['0.670', '0.330']
   가중치: ['0.330', '0.670']
   출력:   ['0.670', '0.330']
   출력:   ['0.330', '0.670']

B) 변환 사용 (W_Q, W_K, W_V):
   가중치: ['0.587', '0.413']
   가중치: ['0.330', '0.670']
   출력:   ['1.587', '1.413']
   출력:   ['1.330', '1.670']
figure saved -> kor/src/images/ch12_1_qkv_weights.svg


## 4. 정리

| 실험 | 관측 | 본문의 어떤 주장을 확인하나 |
|---|---|---|
| `qkv(X, WQ, WK, WV)` | 같은 X에서 Q ≠ K ≠ V | 세 변환 행렬은 독립적인 학습 파라미터 |
| QK^T → softmax → AV | 가중치 (0.588, 0.412)/(0.330, 0.670), 출력은 V의 가중평균 | 어텐션 = "관련도만큼 내용을 섞는다" |
| Q=K=V=X (정체) | 가중치 (0.731, 0.269), 출력 ≈ "자기 자신 + 남의 것 약간" | 변환 없으면 표현력 제한 + 학습 여지 0 |

**다음 12.2절**: scaled dot-product attention의 공식
\(\text{Attention}(Q,K,V)=\text{softmax}(QK^T/\sqrt{d_k})V\)과,
Q·K·V가 모두 같은 문장에서 나온 **self-attention** — "문장이 자기
자신을 본다"가 대명사 지시 해결로 이어지는 과정을 다룹니다.